[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dongzoolee/hidden-bites/blob/main/notebooks/hb-score-factor-explorer.ipynb)

# HB Score Factor Explorer

이 노트북은 `hb-score-calculation.ipynb`의 full run 산출물인 `datasets/derived/hb-score-restaurants.json`을 읽고, Hidden Bites factor를 바꿔가며 top 50 서울 맛집의 score ranking이 어떻게 달라지는지 확인합니다.

NLI 모델은 다시 실행하지 않습니다. 이미 계산된 `restaurants[].scores[factor_id]` 값을 사용하므로 Colab에서도 빠르게 열어볼 수 있습니다.

## 0. 실행 환경 준비

Colab과 로컬 Jupyter 모두에서 같은 widget explorer가 동작하도록 `pandas`, `plotly`, `ipywidgets`만 준비합니다.

In [ ]:
%pip install -q pandas plotly ipywidgets

## 1. HB score 산출물 로드

로컬에서는 repository 안의 `datasets/derived/hb-score-restaurants.json`을 먼저 읽고, Colab에서는 GitHub raw URL을 읽습니다.

In [ ]:
import json
from pathlib import Path
from urllib.request import urlopen

import ipywidgets as widgets
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

RAW_DATA_URL = "https://raw.githubusercontent.com/dongzoolee/hidden-bites/main/datasets/derived/hb-score-restaurants.json"
LOCAL_DATA_PATH = Path("datasets/derived/hb-score-restaurants.json")


def running_in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False


def load_score_data():
    if LOCAL_DATA_PATH.exists() and not running_in_colab():
        return json.loads(LOCAL_DATA_PATH.read_text(encoding="utf-8")), str(LOCAL_DATA_PATH)
    with urlopen(RAW_DATA_URL) as response:
        return json.loads(response.read().decode("utf-8")), RAW_DATA_URL


score_data, data_source = load_score_data()
metadata = score_data.get("metadata", {})
factors = score_data.get("factors", [])
restaurants = score_data.get("restaurants", [])

assert metadata.get("run_mode") == "full"
assert metadata.get("scored_restaurant_count") == 50
assert len(factors) == 10
assert len(restaurants) == 50

print(f"Loaded {len(restaurants)} restaurants and {len(factors)} factors from {data_source}")
metadata

## 2. 그래프용 flat table 구성

JSON의 nested score map을 restaurant-factor 단위 table로 펼칩니다. 한 restaurant가 10개 factor row를 갖습니다.

In [ ]:
FACTOR_OPTIONS = [(factor["label"], factor["id"]) for factor in factors]
FACTOR_LABEL_BY_ID = {factor_id: label for label, factor_id in FACTOR_OPTIONS}
METRIC_OPTIONS = [
    ("HB Score", "hb_score"),
    ("Raw HB Score", "raw_hb_score"),
    ("Mean Factor Relevance", "mean_factor_relevance"),
    ("Mean Factor Distance", "mean_factor_distance"),
]
METRIC_LABEL_BY_ID = {metric_id: label for label, metric_id in METRIC_OPTIONS}
SORT_OPTIONS = [("Selected metric descending", "metric_desc"), ("Google place rank", "place_rank")]

rows = []
for restaurant in restaurants:
    score_map = restaurant.get("scores", {})
    assert set(score_map.keys()) == set(FACTOR_LABEL_BY_ID.keys())
    for factor_id, score in score_map.items():
        hb_score = float(score["hb_score"])
        assert 0.0 <= hb_score <= 5.0
        rows.append(
            {
                "place_rank": int(restaurant["place_rank"]),
                "place_name": restaurant["place_name"],
                "place_id": restaurant["place_id"],
                "google_maps_uri": restaurant.get("google_maps_uri"),
                "formatted_address": restaurant.get("formatted_address"),
                "google_place_rating": restaurant.get("google_place_rating"),
                "popularity_count": int(restaurant.get("popularity_count") or 0),
                "collected_review_count": int(restaurant.get("collected_review_count") or 0),
                "collection_status": restaurant.get("collection_status"),
                "factor_id": factor_id,
                "factor_label": FACTOR_LABEL_BY_ID[factor_id],
                "hb_score": hb_score,
                "raw_hb_score": float(score["raw_hb_score"]),
                "count_bonus": float(score["count_bonus"]),
                "mean_factor_relevance": float(score["mean_factor_relevance"]),
                "mean_factor_distance": float(score["mean_factor_distance"]),
                "scored_review_count": int(score["scored_review_count"]),
            }
        )

score_df = pd.DataFrame(rows)
assert score_df.shape[0] == len(restaurants) * len(factors)
score_df.head()

## 3. Factor별 restaurant score explorer

Factor와 metric을 바꾸면 bar chart와 ranking table이 함께 갱신됩니다. 기본 metric인 `HB Score`는 factor relevance가 반영된 별점 기여도에 popularity bonus를 더한 값입니다.

In [ ]:
def restaurant_label(row):
    return f"#{int(row.place_rank):02d} {row.place_name}"


def filtered_scores(factor_id, metric_id, top_n, sort_mode):
    view = score_df.loc[score_df["factor_id"] == factor_id].copy()
    if sort_mode == "place_rank":
        view = view.sort_values(["place_rank", metric_id], ascending=[True, False])
    else:
        view = view.sort_values([metric_id, "place_rank"], ascending=[False, True])
    view = view.head(int(top_n)).copy()
    view["restaurant_label"] = view.apply(restaurant_label, axis=1)
    return view


def make_score_figure(factor_id="taste", metric_id="hb_score", top_n=20, sort_mode="metric_desc"):
    view = filtered_scores(factor_id, metric_id, top_n, sort_mode)
    plot_view = view.iloc[::-1]
    metric_label = METRIC_LABEL_BY_ID[metric_id]
    factor_label = FACTOR_LABEL_BY_ID[factor_id]
    hovertemplate = (
        "<b>%{customdata[0]}</b><br>"
        "Rank: %{customdata[1]}<br>"
        "Google rating: %{customdata[2]}<br>"
        "Collected reviews: %{customdata[3]}<br>"
        f"{metric_label}: " + "%{x:.4f}<br>"
        "HB Score: %{customdata[4]:.4f}<br>"
        "Raw HB Score: %{customdata[5]:.4f}<br>"
        "Count bonus: %{customdata[6]:.4f}<extra></extra>"
    )
    fig = go.Figure(
        go.Bar(
            x=plot_view[metric_id],
            y=plot_view["restaurant_label"],
            orientation="h",
            marker={"color": plot_view[metric_id], "colorscale": "Viridis", "showscale": False},
            customdata=plot_view[["place_name", "place_rank", "google_place_rating", "collected_review_count", "hb_score", "raw_hb_score", "count_bonus"]],
            hovertemplate=hovertemplate,
        )
    )
    x_range = [0, 5] if metric_id in {"hb_score", "raw_hb_score"} else [0, 1]
    fig.update_layout(
        title=f"{factor_label} restaurant ranking by {metric_label}",
        xaxis={"title": metric_label, "range": x_range},
        yaxis={"title": "Restaurant", "automargin": True},
        height=max(520, 28 * len(plot_view) + 160),
        margin={"l": 260, "r": 40, "t": 80, "b": 60},
        template="plotly_white",
    )
    return fig


def make_score_table(factor_id="taste", metric_id="hb_score", top_n=20, sort_mode="metric_desc"):
    view = filtered_scores(factor_id, metric_id, top_n, sort_mode)
    columns = [
        "place_rank",
        "place_name",
        "factor_label",
        metric_id,
        "hb_score",
        "raw_hb_score",
        "count_bonus",
        "google_place_rating",
        "collected_review_count",
    ]
    table = view[columns].copy()
    for column in [metric_id, "hb_score", "raw_hb_score", "count_bonus", "google_place_rating"]:
        table[column] = table[column].map(lambda value: round(float(value), 4) if pd.notna(value) else value)
    return table


fig = make_score_figure()
assert len(fig.data) == 1
assert len(fig.data[0].x) == 20
make_score_table().head()

In [ ]:
factor_dropdown = widgets.Dropdown(options=FACTOR_OPTIONS, value="taste", description="factor")
metric_dropdown = widgets.Dropdown(options=METRIC_OPTIONS, value="hb_score", description="metric")
top_n_slider = widgets.IntSlider(value=20, min=5, max=50, step=5, description="top n", continuous_update=False)
sort_dropdown = widgets.Dropdown(options=SORT_OPTIONS, value="metric_desc", description="sort")
plot_output = widgets.Output()
table_output = widgets.Output()


def render_explorer(factor_id, metric_id, top_n, sort_mode):
    with plot_output:
        clear_output(wait=True)
        display(make_score_figure(factor_id=factor_id, metric_id=metric_id, top_n=top_n, sort_mode=sort_mode))
    with table_output:
        clear_output(wait=True)
        display(make_score_table(factor_id=factor_id, metric_id=metric_id, top_n=top_n, sort_mode=sort_mode))


def handle_change(change):
    render_explorer(
        factor_id=factor_dropdown.value,
        metric_id=metric_dropdown.value,
        top_n=top_n_slider.value,
        sort_mode=sort_dropdown.value,
    )


for control in [factor_dropdown, metric_dropdown, top_n_slider, sort_dropdown]:
    control.observe(handle_change, names="value")

controls = widgets.HBox([factor_dropdown, metric_dropdown, top_n_slider, sort_dropdown])
display(widgets.VBox([controls, plot_output, table_output]))
handle_change(None)

## 4. 빠른 해석용 pivot table

아래 table은 모든 restaurant의 factor별 `hb_score`를 한 번에 비교하기 위한 정적 요약입니다.

In [ ]:
hb_score_pivot = score_df.pivot_table(index=["place_rank", "place_name"], columns="factor_label", values="hb_score")
hb_score_pivot = hb_score_pivot.reset_index().sort_values("place_rank")
hb_score_pivot.head(50)